# Notebook 10 — Multi-Dataset Raw EEG Clustering Readiness

This notebook implements the data-contract stage of the biodata-informed clustering
strategy. It audits the raw files currently present, defines format readers and
portable feature tiers, and creates a dataset/validation registry.

It does **not** fabricate external data or cross-dataset results. OpenNeuro
`ds005284` and an audited 10-subject subset of PhysioNet `eegmmidb` are stored
locally. The independent EDF state validation is implemented in Notebook/Script 11.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import mne

mne.set_log_level("ERROR")
DATA = Path("data")
PREPROCESSED = DATA / "preprocessed"
CURRENT_ROOT = DATA / "ds005284"

registry = pd.DataFrame([
    {
        "dataset_id": "ds005284",
        "source": "OpenNeuro",
        "url": "https://openneuro.org/datasets/ds005284",
        "expected_format": "BDF + BIDS sidecars",
        "primary_validation_role": "Development baseline",
        "local_status": "available",
        "local_root": str(CURRENT_ROOT),
    },
    {
        "dataset_id": "eegmmidb",
        "source": "PhysioNet",
        "url": "https://physionet.org/content/eegmmidb/",
        "expected_format": "EDF+",
        "primary_validation_role": "Rest/movement/imagery and format transfer",
        "local_status": "available_subset_audited" if (DATA / "eegmmidb").exists() else "not_downloaded",
        "local_root": str(DATA / "eegmmidb") if (DATA / "eegmmidb").exists() else "",
    },
    {
        "dataset_id": "sleep-edfx",
        "source": "PhysioNet",
        "url": "https://physionet.org/content/sleep-edfx/1.0.0/",
        "expected_format": "EDF + hypnogram",
        "primary_validation_role": "Sparse montage and temporal-state validation",
        "local_status": "not_downloaded",
        "local_root": "",
    },
    {
        "dataset_id": "ds004148",
        "source": "OpenNeuro",
        "url": "https://openneuro.org/datasets/ds004148/versions/1.0.1",
        "expected_format": "BIDS EEG; inspect after download",
        "primary_validation_role": "Test-retest rest/cognitive-state stability",
        "local_status": "not_downloaded",
        "local_root": "",
    },
    {
        "dataset_id": "ds003838",
        "source": "OpenNeuro",
        "url": "https://openneuro.org/datasets/ds003838/versions/1.0.6",
        "expected_format": "BIDS EEG + ECG + PPG + pupillometry",
        "primary_validation_role": "Multimodal external validation",
        "local_status": "not_downloaded",
        "local_root": "",
    },
    {
        "dataset_id": "DEAP",
        "source": "Queen Mary University of London",
        "url": "https://www.eecs.qmul.ac.uk/mmv/datasets/deap/",
        "expected_format": "Dataset-specific EEG + peripheral physiology",
        "primary_validation_role": "Multimodal affect sensitivity",
        "local_status": "access_terms_require_review",
        "local_root": "",
    },
])
registry.to_csv(PREPROCESSED / "multidataset_eeg_registry.csv", index=False)
print(registry[["dataset_id", "expected_format", "local_status"]].to_string(index=False))

dataset_id                              expected_format                local_status
  ds005284                          BDF + BIDS sidecars                   available
  eegmmidb                                         EDF+ available_subset_audited
sleep-edfx                              EDF + hypnogram              not_downloaded
  ds004148             BIDS EEG; inspect after download              not_downloaded
  ds003838          BIDS EEG + ECG + PPG + pupillometry              not_downloaded
      DEAP Dataset-specific EEG + peripheral physiology access_terms_require_review


## Generic MNE raw reader

This dispatch supports common EEG formats. BIDS metadata and dataset-specific channel
mapping still require explicit adapters; file readability alone is not
harmonization.

In [2]:
def read_raw_header(path):
    path = Path(path)
    suffix = path.suffix.lower()
    readers = {
        ".bdf": mne.io.read_raw_bdf,
        ".edf": mne.io.read_raw_edf,
        ".vhdr": mne.io.read_raw_brainvision,
        ".set": mne.io.read_raw_eeglab,
        ".fif": mne.io.read_raw_fif,
    }
    if suffix not in readers:
        raise ValueError(f"Unsupported raw EEG format: {suffix}")
    return readers[suffix](path, preload=False, verbose="ERROR")

print("Supported extensions:", ".bdf, .edf, .vhdr, .set, .fif")

Supported extensions: .bdf, .edf, .vhdr, .set, .fif


## Audit every local BDF and BIDS sidecar

The BioSemi files store generic channel labels. The accompanying `channels.tsv`
ordering is checked and used to confirm the C3/CZ/C4 mapping requirement. Signal
samples are not loaded during this header audit.

In [3]:
inventory_rows = []
bdf_files = sorted(CURRENT_ROOT.glob("sub-*/eeg/*_eeg.bdf"))

for bdf_path in bdf_files:
    subject = bdf_path.parts[-3]
    stem = bdf_path.name.replace("_eeg.bdf", "")
    channels_path = bdf_path.with_name(stem + "_channels.tsv")
    events_path = bdf_path.with_name(stem + "_events.tsv")
    eeg_json_path = bdf_path.with_name(stem + "_eeg.json")

    raw = read_raw_header(bdf_path)
    sidecar_channels = pd.read_csv(channels_path, sep="\t")
    mapped_names = sidecar_channels["name"].astype(str).tolist()
    mapped_upper = {name.upper() for name in mapped_names}
    eeg_channel_count = sum(channel_type == "eeg" for channel_type in raw.get_channel_types())

    inventory_rows.append({
        "dataset_id": "ds005284",
        "subject": subject,
        "raw_file": str(bdf_path),
        "format": "BDF",
        "file_bytes": bdf_path.stat().st_size,
        "sampling_rate_hz": raw.info["sfreq"],
        "duration_seconds": raw.n_times / raw.info["sfreq"],
        "raw_signal_count": len(raw.ch_names),
        "raw_eeg_channel_count": eeg_channel_count,
        "channels_sidecar_rows": len(sidecar_channels),
        "has_events_tsv": events_path.exists(),
        "has_eeg_json": eeg_json_path.exists(),
        "has_C3": "C3" in mapped_upper,
        "has_CZ": "CZ" in mapped_upper,
        "has_C4": "C4" in mapped_upper,
        "requires_positional_channel_rename": raw.ch_names[:3] != mapped_names[:3],
    })
    raw.close()

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(PREPROCESSED / "current_raw_eeg_inventory.csv", index=False)

assert len(inventory) == 26
assert inventory[["has_events_tsv", "has_eeg_json", "has_C3", "has_CZ", "has_C4"]].all().all()
assert inventory["requires_positional_channel_rename"].all()

print(f"Audited raw files: {len(inventory)}")
print(f"Total BDF bytes: {inventory.file_bytes.sum():,}")
print("Sampling rates:", sorted(inventory.sampling_rate_hz.unique()))
print("Raw signal counts:", sorted(inventory.raw_signal_count.unique()))
print("Durations (seconds):",
      round(inventory.duration_seconds.min(), 2), "to",
      round(inventory.duration_seconds.max(), 2))
print("All files require positional channel renaming:",
      inventory.requires_positional_channel_rename.all())

Audited raw files: 26
Total BDF bytes: 1,783,981,056
Sampling rates: [np.float64(1024.0), np.float64(2048.0)]
Raw signal counts: [np.int64(65)]
Durations (seconds): 325.0 to 346.0
All files require positional channel renaming: True


## Portable multi-dataset feature contract

QC variables are retained for audits but excluded from physiological clustering.
Exact electrode features are a sensitivity tier, not the universal representation.

In [4]:
feature_contract = pd.DataFrame([
    ["portable", "relative_bandpower_spatial_summary", "median/IQR across valid EEG channels", 1, "MNE/SciPy", True],
    ["portable", "spectral_exponent_offset", "median/IQR across channels", 1, "specparam or validated implementation", True],
    ["portable", "entropy_hjorth_summary", "median/IQR across channels", 1, "AntroPy/NumPy", True],
    ["portable", "robust_amplitude_shape", "RMS/PTP/line-length robust summaries", 1, "NumPy", True],
    ["geometry", "global_field_power", "sample/window GFP distributions", 4, "MNE/NumPy", True],
    ["geometry", "microstate_maps", "polarity-invariant GFP-peak topographies", 19, "PyCrostates/MICROSTATELAB method", False],
    ["geometry", "covariance_tangent", "SPD covariance tangent-space features", 4, "PyRiemann", False],
    ["geometry", "regional_connectivity", "coarse region-pair connectivity", 8, "MNE-Connectivity", True],
    ["temporal", "state_occupancy", "fraction of valid recording in state", 1, "NumPy/pandas", True],
    ["temporal", "dwell_and_transition", "dwell distribution/transition entropy", 1, "hmmlearn/NumPy", False],
    ["peripheral", "ecg_hrv", "quality-controlled NN features", 0, "NeuroKit2/pyHRV", False],
    ["peripheral", "ppg_prv", "quality-controlled pulse interval features", 0, "NeuroKit2/HeartPy", False],
    ["peripheral", "eda_features", "tonic/phasic/response features", 0, "NeuroKit2/BioSPPy", False],
    ["peripheral", "respiration_features", "rate/variability/phase", 0, "NeuroKit2/BioSPPy", False],
    ["peripheral", "skin_temperature", "level and within-person deviation", 0, "pandas/NumPy", True],
    ["qc_only", "artifact_metrics", "EOG/EMG/line-noise/motion/bad-channel metrics", 0, "MNE/Autoreject/ICLabel", False],
    ["qc_only", "provenance_metrics", "device/reference/rate/duration/missingness", 0, "BIDS/MNE-BIDS", True],
], columns=[
    "tier", "feature_group", "aggregation", "minimum_eeg_channels",
    "preferred_open_source", "available_with_current_environment"
])
feature_contract.to_csv(PREPROCESSED / "multidataset_feature_contract.csv", index=False)
print(feature_contract.to_string(index=False))

      tier                      feature_group                                   aggregation  minimum_eeg_channels                 preferred_open_source  available_with_current_environment
  portable relative_bandpower_spatial_summary          median/IQR across valid EEG channels                     1                             MNE/SciPy                                True
  portable           spectral_exponent_offset                    median/IQR across channels                     1 specparam or validated implementation                                True
  portable             entropy_hjorth_summary                    median/IQR across channels                     1                         AntroPy/NumPy                                True
  portable             robust_amplitude_shape          RMS/PTP/line-length robust summaries                     1                                 NumPy                                True
  geometry                 global_field_power               

## Compatibility of the existing 135 features

Every current predictor is derived from C3, CZ, C4, or their exact channel pairs.
Therefore the table remains useful for Zhao regression tests but cannot be treated
as the universal input schema.

In [5]:
dictionary = pd.read_csv(PREPROCESSED / "feature_dictionary_135.csv")
dictionary["cross_dataset_role"] = np.where(
    dictionary["feature_family"].eq("Connectivity"),
    "exact-pair sensitivity feature",
    "exact-channel sensitivity feature",
)
dictionary["portable_without_C3_CZ_C4"] = False
compatibility = (
    dictionary.groupby(["feature_family", "cross_dataset_role"], as_index=False)
    .agg(feature_count=("feature_name", "size"))
)
compatibility.to_csv(PREPROCESSED / "current_feature_portability_audit.csv", index=False)
print(compatibility.to_string(index=False))
print("\nPortable current predictors without C3/CZ/C4: 0 of 135")

      feature_family                cross_dataset_role  feature_count
          Band power exact-channel sensitivity feature             45
    Band-power ratio exact-channel sensitivity feature              9
        Connectivity    exact-pair sensitivity feature             30
Nonlinear/complexity exact-channel sensitivity feature             30
      Spectral shape exact-channel sensitivity feature              6
         Time domain exact-channel sensitivity feature             15

Portable current predictors without C3/CZ/C4: 0 of 135


## Validation protocol

State discovery and state meaning are separated. External task, sleep, behavioral or
peripheral variables are used only after unsupervised fitting.

In [6]:
validation_protocol = pd.DataFrame([
    ["split", "Subject and dataset groups", "No random-window leakage"],
    ["normalization", "Fit on training datasets only", "Prevent held-out distribution leakage"],
    ["discovery", "K-Means/GMM/HDBSCAN/HMM candidates", "Do not rely on one cluster geometry"],
    ["stability", "Subject, temporal-block and dataset bootstrap", "Require reproducible co-assignment"],
    ["state_count", "Stability + held-out likelihood + coverage", "Silhouette/elbow alone are insufficient"],
    ["assignment", "KNN/prototype/posterior without refitting", "Measure transport to held-out datasets"],
    ["domain_audit", "Predict subject/dataset/device/QC from states", "Reject nuisance-dominated states"],
    ["external_validation", "Task/sleep/peripheral/behavior after fitting", "Do not manufacture state meaning"],
    ["reporting", "Coverage, entropy, dwell, transitions, uncertainty", "Avoid forced confident labels"],
], columns=["stage", "required_method", "reason"])
validation_protocol.to_csv(
    PREPROCESSED / "multidataset_validation_protocol.csv", index=False
)
print(validation_protocol.to_string(index=False))

              stage                                    required_method                                  reason
              split                         Subject and dataset groups                No random-window leakage
      normalization                      Fit on training datasets only   Prevent held-out distribution leakage
          discovery                 K-Means/GMM/HDBSCAN/HMM candidates     Do not rely on one cluster geometry
          stability      Subject, temporal-block and dataset bootstrap      Require reproducible co-assignment
        state_count         Stability + held-out likelihood + coverage Silhouette/elbow alone are insufficient
         assignment          KNN/prototype/posterior without refitting  Measure transport to held-out datasets
       domain_audit      Predict subject/dataset/device/QC from states        Reject nuisance-dominated states
external_validation       Task/sleep/peripheral/behavior after fitting        Do not manufacture state meaning
 

## Readiness decision

Cross-dataset clustering requires at least two independently acquired local raw EEG
datasets. The current inventory is checked programmatically. A complete audited
subset counts as locally available; task annotations remain excluded from discovery.

In [7]:
available_raw_datasets = registry.loc[
    registry["local_status"].str.startswith("available"), "dataset_id"
].tolist()
ready = len(available_raw_datasets) >= 2

print("Available local raw EEG datasets:", available_raw_datasets)
print("Cross-dataset clustering ready:", ready)
assert ready, "Two independently acquired local raw EEG datasets are required." 

Available local raw EEG datasets: ['ds005284', 'eegmmidb']
Cross-dataset clustering ready: True


## Independent EDF acquisition and state-validation audit

PhysioNet EEG Motor Movement/Imagery v1.0.0 was checked before acquisition: public
access under Open Data Commons Attribution License v1.0, 3.4 GB uncompressed and
1.9 GB as a ZIP. The local preregistered subset contains S001-S010 and runs
01/02/03/04/07/08/11/12 (80 EDF+ files, 167,230,368 bytes).

Portable Tier 1 features and QC were extracted at 2, 4 and 8 seconds by
`11_independent_edf_state_validation.py`. Task/run annotations were retained only
for post-hoc interpretation and were excluded from scaling, clustering, state-count
selection, confound testing and leave-one-dataset-out assignment.

In [8]:
raw_inventory = pd.read_csv(PREPROCESSED / "multidataset_raw_eeg_inventory.csv")
confounds = pd.read_csv(PREPROCESSED / "multidataset_state_confound_audit.csv")
lodo = pd.read_csv(PREPROCESSED / "multidataset_lodo_summary.csv")
conclusion = json.loads(
    (PREPROCESSED / "multidataset_state_validation_conclusion.json").read_text()
)
print("Audited recordings by dataset:", raw_inventory.groupby("dataset_id").size().to_dict())
print("Confound-gate pass by window:", confounds.groupby("window_seconds").accepted.all().to_dict())
print("Minimum bidirectional LODO coverage:", lodo.groupby("window_seconds").coverage.min().round(4).to_dict())
print("Confirmed on one independent EDF dataset:", conclusion["confirmed_on_one_independent_edf_dataset"])
assert not conclusion["confirmed_on_one_independent_edf_dataset"]

Audited recordings by dataset: {'ds005284': 26, 'eegmmidb': 80}
Confound-gate pass by window: {2: True, 4: False, 8: False}
Minimum bidirectional LODO coverage: {2: 0.2355, 4: 0.1651, 8: 0.1803}
Confirmed on one independent EDF dataset: False


## Generated files

- `data/preprocessed/multidataset_eeg_registry.csv`
- `data/preprocessed/current_raw_eeg_inventory.csv`
- `data/preprocessed/multidataset_feature_contract.csv`
- `data/preprocessed/current_feature_portability_audit.csv`
- `data/preprocessed/multidataset_validation_protocol.csv`
- `data/preprocessed/eegmmidb_acquisition_audit.json`
- `data/preprocessed/multidataset_raw_eeg_inventory.csv`
- `data/preprocessed/multidataset_tier1_features_{2,4,8}s.csv`
- `data/preprocessed/multidataset_state_model_selection.csv`
- `data/preprocessed/multidataset_state_confound_audit.csv`
- `data/preprocessed/multidataset_lodo_summary.csv`
- `data/preprocessed/multidataset_lodo_temporal_metrics.csv`
- `data/preprocessed/eegmmidb_posthoc_state_interpretation.csv`
- `data/preprocessed/multidataset_state_validation_conclusion.json`

These are real inventory and strategy outputs. The PhysioNet subset contains
subjects S001-S010 and runs 01/02/03/04/07/08/11/12 under ODC Attribution v1.0.